# Lecture 1: Strings and the Genome — SOLUTIONS

**Course:** Intro to Programming for Computational Biology  
**For instructors only — do not distribute to students.**

---
## Section 1: DNA as a Python String

In [ ]:
dna = "ATGGTGCATCTGACTCCTGAGGAGAAGTCT"
print("Sequence:", dna)
print("Length:", len(dna))

### Exercise 1.1 — Extract the 4th codon

In [ ]:
fourth_codon = dna[9:12]
print("4th codon:", fourth_codon)  # CTG

### Exercise 1.2 — GC content

In [ ]:
def gc_content(sequence):
    """Return the GC content of a DNA sequence as a value between 0 and 1."""
    gc = sequence.count("G") + sequence.count("C")
    return gc / len(sequence)

print(gc_content(dna))  # ~0.567

### Exercise 1.3 — Split into codons

In [ ]:
codons = []
for i in range(0, len(dna), 3):
    codon = dna[i:i+3]
    codons.append(codon)

print(codons)
# ['ATG', 'GTG', 'CAT', 'CTG', 'ACT', 'CCT', 'GAG', 'GAG', 'AAG', 'TCT']

---
## Section 2: From DNA to Protein

In [ ]:
mrna = dna.replace("T", "U")
print("mRNA:", mrna)

In [ ]:
codon_table = {
    "UUU": "F", "UUC": "F",
    "UUA": "L", "UUG": "L", "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "AUU": "I", "AUC": "I", "AUA": "I",
    "AUG": "M",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "AGU": "S", "AGC": "S",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "UAU": "Y", "UAC": "Y",
    "UAA": "*", "UAG": "*", "UGA": "*",
    "CAU": "H", "CAC": "H",
    "CAA": "Q", "CAG": "Q",
    "AAU": "N", "AAC": "N",
    "AAA": "K", "AAG": "K",
    "GAU": "D", "GAC": "D",
    "GAA": "E", "GAG": "E",
    "UGU": "C", "UGC": "C",
    "UGG": "W",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R", "AGA": "R", "AGG": "R",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

### Exercise 2.1 — Translate

In [ ]:
def translate(dna_sequence):
    """Translate a DNA sequence into a protein sequence."""
    mrna = dna_sequence.replace("T", "U")
    protein = ""
    for i in range(0, len(mrna), 3):
        codon = mrna[i:i+3]
        amino_acid = codon_table.get(codon, "?")
        if amino_acid == "*":
            break
        protein += amino_acid
    return protein

print(translate(dna))  # MVHLTP EEKS

---
## Section 3: Introducing Biopython

In [ ]:
from Bio.Seq import Seq
from Bio import Entrez, SeqIO

Entrez.email = "your.email@example.com"

In [ ]:
seq = Seq(dna)
print("Complement:        ", seq.complement())
print("Reverse complement:", seq.reverse_complement())
print("Transcription:     ", seq.transcribe())
print("Translation:       ", seq.translate())

### Exercise 3.1 — Compare Biopython to manual translation

In [ ]:
bio_protein = str(seq.translate())
bio_protein_clean = bio_protein[:-1]  # Remove trailing *

manual_result = translate(dna)
print("Biopython:", bio_protein_clean)
print("Manual:   ", manual_result)
print("Match:", bio_protein_clean == manual_result)

In [ ]:
# Fetch HBB from NCBI
handle = Entrez.efetch(db="nucleotide", id="NM_000518", rettype="gb", retmode="text")
hbb_record = SeqIO.read(handle, "genbank")
handle.close()

print("Gene:", hbb_record.name)
print("Description:", hbb_record.description)
print("Sequence length:", len(hbb_record.seq), "bp")
print("First 60 bp:", hbb_record.seq[:60])

In [ ]:
for feature in hbb_record.features:
    if feature.type == "CDS":
        hbb_cds = feature.extract(hbb_record.seq)
        break

print("CDS length:", len(hbb_cds), "bp")
print("CDS:", hbb_cds)

In [ ]:
hbb_protein = hbb_cds.translate(to_stop=True)
print("HBB protein (β-globin):")
print(hbb_protein)
print("Length:", len(hbb_protein), "amino acids")

### Exercise 3.2 — Count glutamate residues

In [ ]:
num_E = str(hbb_protein).count("E")
print("Glutamate (E) count:", num_E)
print("Amino acid at position 6:", str(hbb_protein)[5])  # index 5 = position 6

---
## Section 4: Mutations

In [ ]:
def point_mutation(sequence, position, new_base):
    return sequence[:position] + new_base + sequence[position + 1:]

hbb_cds_str = str(hbb_cds)
print("Codon 6 (normal):  ", hbb_cds_str[15:18])  # GAG

hbb_sickle = point_mutation(hbb_cds_str, 16, "T")
print("Codon 6 (sickle):  ", hbb_sickle[15:18])   # GTG

In [ ]:
normal_protein = Seq(hbb_cds_str).translate(to_stop=True)
sickle_protein = Seq(hbb_sickle).translate(to_stop=True)

print("Normal position 6:  ", normal_protein[5])   # E
print("Sickle position 6:  ", sickle_protein[5])   # V
print("Proteins identical? ", normal_protein == sickle_protein)

### Exercise 4.1 — classify_mutation

In [ ]:
def classify_mutation(original_cds, mutated_cds):
    original_protein = str(Seq(original_cds).translate())
    mutated_protein  = str(Seq(mutated_cds).translate())

    if "*" in mutated_protein and mutated_protein.index("*") < original_protein.index("*"):
        return "nonsense"
    elif original_protein != mutated_protein:
        return "missense"
    else:
        return "synonymous"

result = classify_mutation(hbb_cds_str, hbb_sickle)
print("Sickle cell mutation is:", result)  # missense

### Exercise 4.2 — Design your own synonymous and nonsense mutations

**Sample answers** (many valid choices exist):

In [ ]:
# Synonymous: GAG (E) → GAA (E) at codon 6 — change position 17, G→A
hbb_syn = point_mutation(hbb_cds_str, 17, "A")
print("Codon 6 synonymous:", hbb_syn[15:18])  # GAA
print("Classification:", classify_mutation(hbb_cds_str, hbb_syn))

# Nonsense: GAG (E) → TAG (stop) at codon 6 — change position 15, G→T
hbb_nonsense = point_mutation(hbb_cds_str, 15, "T")
print("\nCodon 6 nonsense:", hbb_nonsense[15:18])  # TAG
print("Classification:", classify_mutation(hbb_cds_str, hbb_nonsense))

### Exercise 4.3 — Real HBB variants

In [ ]:
# HbC: position 16, A→G (GAG→GGG, E→G = missense)
hbb_HbC = point_mutation(hbb_cds_str, 16, "G")
print("HbC codon 6:", hbb_HbC[15:18])  # GGG
print("Classification:", classify_mutation(hbb_cds_str, hbb_HbC))
# HbC causes a milder hemolytic anemia; heterozygotes have some protection against malaria

# HbE: position 76, G→A (GAG→AAG, E→K = missense at codon 26)
hbb_HbE = point_mutation(hbb_cds_str, 76, "A")
print("\nHbE codon 26:", hbb_HbE[75:78])
print("Classification:", classify_mutation(hbb_cds_str, hbb_HbE))
# HbE is very common in Southeast Asia; also causes mild hemolytic anemia

---
## Section 5: Advanced Topics — Sample Solutions

### 5.1 HGVS notation parser

In [ ]:
import re

def parse_hgvs(hgvs_string):
    """
    Parse a simple HGVS coding sequence variant string.
    Example input: 'c.17A>T'
    Returns: (position_0based, ref_base, alt_base)
    """
    match = re.match(r"c\.([0-9]+)([ACGT])>([ACGT])", hgvs_string)
    if not match:
        raise ValueError(f"Cannot parse HGVS: {hgvs_string}")
    pos_1based = int(match.group(1))
    ref = match.group(2)
    alt = match.group(3)
    return pos_1based - 1, ref, alt  # convert to 0-based

pos, ref, alt = parse_hgvs("c.17A>T")
print(f"Position (0-based): {pos}, {ref}→{alt}")

# Apply to HBB
hbb_mut = point_mutation(hbb_cds_str, pos, alt)
print("Classification:", classify_mutation(hbb_cds_str, hbb_mut))

### 5.2 Motif finding

In [ ]:
import re

def find_motif(sequence, motif):
    """
    Find all start positions (0-based) of a motif in a sequence.
    N in the motif matches any base.
    """
    pattern = motif.replace("N", "[ACGT]")
    return [m.start() for m in re.finditer(f"(?={pattern})", str(sequence))]

# Fetch 500 bp upstream of HBB CDS as a proxy for promoter
for feature in hbb_record.features:
    if feature.type == "CDS":
        cds_start = int(feature.location.start)
        break

promoter = hbb_record.seq[max(0, cds_start - 500): cds_start]
hits = find_motif(promoter, "GCCNCC")
print(f"Found {len(hits)} hits for GCCNCC in HBB promoter at positions:", hits)